# Étape 1 (v2) — Multi-Factor Preprocessing Notebook
## MASI Hybrid Forecasting System

**Input:** `data/interim/masi_merged.csv` (4,786 obs, 2007-2026, 13 raw cols)
**Pipeline:** `scripts/01_preprocessing.py`
**Anti-leakage rules enforced:** L1, L4, L6, L7

This notebook **imports** the pipeline functions from the `.py` script (no duplication).

## 1. Setup and run pipeline

In [ ]:
import sys, os
from pathlib import Path
import importlib.util as _ilu

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

_spec = _ilu.spec_from_file_location('e1', PROJECT_ROOT / 'scripts' / '01_preprocessing.py')
e1 = _ilu.module_from_spec(_spec)
sys.modules['e1'] = e1
_spec.loader.exec_module(e1)
run_pipeline = e1.run_pipeline
COLUMNS_TO_SCALE = e1.COLUMNS_TO_SCALE

pd.set_option('display.max_columns', 16)
%matplotlib inline
plt.rcParams['figure.dpi'] = 100

In [ ]:
artifacts = run_pipeline(verbose=True)

## 2. Inspect splits

In [ ]:
print('Splits:')
for name in ('train', 'val', 'test'):
    df = artifacts[name]
    print(f'  {name:5s}: {len(df):5d} obs  {df.index.min().date()} -> {df.index.max().date()}')

print('\nTrain head:')
display(artifacts['train'].head())

print('\nColumns in train:')
print(list(artifacts['train'].columns))

## 3. Temporal split visualization

In [ ]:
Image(artifacts['plot_paths']['split_overview'])

## 4. Multi-factor inputs (10 raw series)

In [ ]:
Image(artifacts['plot_paths']['factor_overview'])

## 5. Target distribution per partition

In [ ]:
Image(artifacts['plot_paths']['target_distribution'])

In [ ]:
import scipy.stats as sst
for name, part in [('TRAIN', artifacts['train']), ('VAL', artifacts['val']), ('TEST', artifacts['test'])]:
    y = part['target_y_next'].dropna()
    print(f'{name:6s}  mu={y.mean():+.6f}  sigma={y.std():.6f}  skew={sst.skew(y):+.3f}  kurt={sst.kurtosis(y):+.3f}  n={len(y)}')

ks_tv = sst.ks_2samp(artifacts['train']['target_y_next'].dropna(), artifacts['val']['target_y_next'].dropna())
ks_tt = sst.ks_2samp(artifacts['train']['target_y_next'].dropna(), artifacts['test']['target_y_next'].dropna())
print(f'\nKS train vs val:  stat={ks_tv.statistic:.4f}  p={ks_tv.pvalue:.4f}')
print(f'KS train vs test: stat={ks_tt.statistic:.4f}  p={ks_tt.pvalue:.4f}')

## 6. Scaler diagnostic (L1 verification)

After fitting on TRAIN only:
- TRAIN distribution should be μ≈0, σ≈1
- VAL and TEST may show drift — this is **expected** and **informative** (regime shifts)

In [ ]:
Image(artifacts['plot_paths']['scaler_diagnostic'])

In [ ]:
import json
stats_path = os.path.join(os.path.dirname(artifacts['csv_paths']['train']), 'scaler_stats_train_only.json')
with open(stats_path) as f:
    stats = json.load(f)
stats_df = pd.DataFrame(stats).T[['feature', 'mean', 'std', 'n_train']]
display(stats_df)

## 7. Anti-leakage sanity checks

In [ ]:
train = artifacts['train']; val = artifacts['val']; test = artifacts['test']

print('--- L6: temporal ordering ---')
print(f'  train end:  {train.index.max().date()}')
print(f'  val start:  {val.index.min().date()}  gap = {(val.index.min() - train.index.max()).days} cal days')
print(f'  val end:    {val.index.max().date()}')
print(f'  test start: {test.index.min().date()}  gap = {(test.index.min() - val.index.max()).days} cal days')

print('\n--- L4: target after features ---')
print(f'  target_y_next present? {"target_y_next" in train.columns}')
print(f'  no NaN at end of train? {not pd.isna(train["target_y_next"].iloc[-1])}')

print('\n--- L1: scaler stats match TRAIN ---')
for c in ['masi_close', 'log_return']:
    print(f'  {c:12s}: train mean={train[c].mean():.6f}  scaler stat={stats[c]["mean"]:.6f}  (must match)')

## Summary

| Item | Value |
|------|-------|
| Source | `data/interim/masi_merged.csv` |
| Total clean obs | 4,784 |
| TRAIN | 3,348 obs (2007-02-01 → 2020-07-09) — includes 2008 + COVID 2020 |
| VAL | 478 obs (2020-07-17 → 2022-06-20) |
| TEST | 948 obs (2022-06-28 → 2026-04-17) |
| Target | `target_y_next = ln(P_{t+1}/P_t)` on `masi_close` |
| Scaled features | 10 inputs + log_return |
| Anti-leakage | L1 ✅ L4 ✅ L6 ✅ L7 ✅ |

**Next:** Étape 2 — Mandatory Baselines (RW, Historical Mean, ARIMA, Random Forest).
DL is FORBIDDEN until all 4 baselines are reported (per `prompt.md` RULE 8).